In [1]:
import pandas as pd
import numpy as np
import zipfile
from datetime import datetime
import os
pd.set_option("display.max_colwidth", None)



In [23]:
espacios_col = [(0, 3), (3, 23), (23, 43), (43,45), (45, 48), (48, 49), (368, 376), (376, 384), (384, 392), (346, 347), (396, 411), (411, 426)
                ]

column_names = ["Tipo de seguro", "Certificado", "Numero Interno Del Canal", "Tipo de Registro", "Moneda", 
                "Tipo de Movimiento", "Fecha de Afiliacion", "Fecha de inicio del seguro", "Fecha fin del seguro",
                "Periodo de pago", "Monto Asegurado", "Prima"
                ]

In [24]:
def Extraer_fechas(filename):
    try:
        base = os.path.splitext(os.path.basename(filename))[0]
        # Primera fecha (ddmmyy → fecha completa)
        date_str = base[3:9]   # "030120"
        fecha_trama = datetime.strptime(date_str, "%d%m%y").date()

        # Segunda fecha (ddmm → usar año de la primera fecha)
        decl_str = base[10:14]  # "0601"
        dia, mes = int(decl_str[:2]), int(decl_str[2:])
        fecha_declarada = datetime(fecha_trama.year, mes, dia).date()

        return fecha_trama, fecha_declarada
    except Exception:
        return None, None  # Si algo falla


In [25]:
def Convertir_fecha(columna):
    col = columna.astype(str) # Convertir a string
    col = col.str.strip()  # Eliminar espacios
    # Quitar decimales tipo '19880329.0'
    col = col.str.replace(r"\.0$", "", regex=True)
    col = col.str[:8] # Mantener solo los 8 primeros caracteres
    # Reemplazar valores inválidos (que no tengan 8 dígitos) por NaN
    col = col.where(col.str.match(r"^\d{8}$"), np.nan)
    return pd.to_datetime(col, format="%Y%m%d", errors="coerce")  # Convertir a fecha

In [26]:
# Convertir cada línea en un dict según los cortes
def Convertir_linea(linea):
    row= {name: linea[start:end].strip() for (start, end), name in zip(espacios_col, column_names)}
    row["Trama Original"] = linea.rstrip("\n")       
    return row

In [ ]:
archivo_zip = []
with zipfile.ZipFile('C:/data/TRAMAS/RED_2020-01.zip', "r") as archivo_zip:
    tramas_txt = archivo_zip.namelist()
    #for file in tramas_txt:
        #with archivo_zip.open(file) as file_txt:
        #    file_content = file_txt.read()
        #    list_lines = file_content.decode('latin-1').splitlines()
    file_content= archivo_zip.read(tramas_txt[2])
    lista_lineas = file_content.decode('latin-1').splitlines()
    print(lista_lineas[0:3])
print(tramas_txt)


['703001101013140001502280101                00PEN4                                                                                                                                                                                                                                                                                                         M                     202001062019122720200127000{00000000600000{00000000000480I                                                                                                                                                                                                                                                                                          ', '703001101021840002808010102                00PEN4                                                                                                                                                                                                                                             

In [27]:
with open("C:/data/M_SUSTCRIS100_DESGRAVAMEN.txt", "r", encoding="latin-1") as file:
    lista_lineas =  file.readlines()

In [28]:
df_tramas = pd.DataFrame([Convertir_linea(linea) for linea in lista_lineas])

In [29]:
df_tramas.head(3)

,Tipo de seguro,Certificado,Numero Interno Del Canal,Tipo de Registro,Moneda,Tipo de Movimiento,Fecha de Afiliacion,Fecha de inicio del seguro,Fecha fin del seguro,Periodo de pago,Monto Asegurado,Prima,Trama Original
0,901,00110002324000105283,00110057710219211849,01,PEN,4,,20200406,20200506,M,000000000000000,000000000001400,901001100023240001052830011005771021921184901PEN4 M 20200406202005060000000000000000000000000000001400 02
1,901,00110002324000105283,00110057710219211849,01,PEN,4,,20200505,20200605,M,000000000000000,000000000001400,901001100023240001052830011005771021921184901PEN4 M 20200505202006050000000000000000000000000000001400 02
2,901,00110002324000105283,00110057710219211849,01,PEN,4,,20200605,20200705,M,000000000000000,000000000001400,901001100023240001052830011005771021921184901PEN4 M 20200605202007050000000000000000000000000000001400 02


In [31]:
letra = np.array(list("ABCDEFGHIJKLMNOPQRSTUVWXYZ{0123456789"))
valor = np.array(list("1234567891234567890000000000123456789"))
diccionario_reemplazo = dict(zip(letra, valor))
def convertir_prima(valor_str):
    try:
        # Debe ser al menos 2 caracteres (números + letra)
        if not isinstance(valor_str, str) or len(valor_str) < 2:
            return np.nan
        
        parte_numerica = valor_str[:-1]
        letra_final = valor_str[-1]
        # Buscar equivalencia
        digito = diccionario_reemplazo.get(letra_final)
        if digito is None:
            return np.nan  # letra desconocida → nulo
        # Construir número completo
        numero_str = parte_numerica + digito
        # Intentar convertir a Decimal
        return float(numero_str) / 100

    except Exception:
        return np.nan  # En caso de cualquier error, devolver nulo


In [32]:
df_tramas["Monto Asegurado"] = df_tramas["Monto Asegurado"].apply(convertir_prima)
df_tramas["Prima Bruta"] = df_tramas["Prima"].apply(convertir_prima)
df_tramas["Error Prima"] = df_tramas["Prima Bruta"].isna()

In [33]:
df_tramas["Fecha de Afiliacion"] = Convertir_fecha(df_tramas["Fecha de Afiliacion"])
df_tramas["Fecha de inicio del seguro"] = Convertir_fecha(df_tramas["Fecha de inicio del seguro"])
df_tramas["Fecha fin del seguro"] = Convertir_fecha(df_tramas["Fecha fin del seguro"])

In [132]:
fecha_trama, fecha_declarada = Extraer_fechas(tramas_txt[2])
df_tramas["Fecha Trama"] = fecha_trama
df_tramas["Fecha Declarada"] = fecha_declarada
df_tramas['Fecha Trama'] = pd.to_datetime(df_tramas['Fecha Trama'], errors='coerce')
df_tramas['Fecha Declarada'] = pd.to_datetime(df_tramas['Fecha Declarada'], errors='coerce')

In [34]:
df_tramas.columns = (df_tramas.columns
                     .str.strip()  # quitar espacios al inicio/fin
                     .str.upper()  # opcional: todo en mayúsculas
                     .str.replace(r'[^A-Za-z0-9]', '_', regex=True)  # reemplazar todo lo que no sea letra/número/por _
)

In [35]:
df_tramas.head()

,TIPO_DE_SEGURO,CERTIFICADO,NUMERO_INTERNO_DEL_CANAL,TIPO_DE_REGISTRO,MONEDA,TIPO_DE_MOVIMIENTO,FECHA_DE_AFILIACION,FECHA_DE_INICIO_DEL_SEGURO,FECHA_FIN_DEL_SEGURO,PERIODO_DE_PAGO,MONTO_ASEGURADO,PRIMA,TRAMA_ORIGINAL,PRIMA_BRUTA,ERROR_PRIMA
0,901,00110002324000105283,00110057710219211849,01,PEN,4,NaT,2020-04-06,2020-05-06,M,0.0,000000000001400,901001100023240001052830011005771021921184901PEN4 M 20200406202005060000000000000000000000000000001400 02,14.0,False
1,901,00110002324000105283,00110057710219211849,01,PEN,4,NaT,2020-05-05,2020-06-05,M,0.0,000000000001400,901001100023240001052830011005771021921184901PEN4 M 20200505202006050000000000000000000000000000001400 02,14.0,False
2,901,00110002324000105283,00110057710219211849,01,PEN,4,NaT,2020-06-05,2020-07-05,M,0.0,000000000001400,901001100023240001052830011005771021921184901PEN4 M 20200605202007050000000000000000000000000000001400 02,14.0,False
3,901,00110002324000105283,00110057710219211849,01,PEN,4,NaT,2020-07-06,2020-08-06,M,0.0,000000000001400,901001100023240001052830011005771021921184901PEN4 M 20200706202008060000000000000000000000000000001400 02,14.0,False
4,901,00110002324000105283,00110057770252744798,01,PEN,4,NaT,2020-08-05,2020-09-05,M,0.0,000000000001400,901001100023240001052830011005777025274479801PEN4 M 20200805202009050000000000000000000000000000001400 02,14.0,False


In [36]:
df_tramas['MONTO_ASEGURADO'].value_counts()

MONTO_ASEGURADO
0.00         3737
8650.80        20
19795.00        6
119851.18       2
120695.57       2
             ... 
53695.22        1
54210.02        1
54737.69        1
7990.63         1
8229.93         1
Name: count, Length: 442, dtype: int64

In [18]:
df_filtrado = df_tramas[df_tramas["PRIMA_BRUTA"].isna()]
df_filtrado.head(3)

,TIPO_DE_SEGURO,CERTIFICADO,NUMERO_INTERNO_DEL_CANAL,TIPO_DE_REGISTRO,MONEDA,TIPO_DE_MOVIMIENTO,FECHA_DE_AFILIACION,FECHA_DE_INICIO_DEL_SEGURO,FECHA_FIN_DEL_SEGURO,PERIODO_DE_PAGO,PRIMA,TRAMA_ORIGINAL,PRIMA_BRUTA,ERROR_PRIMA


In [19]:
df_tramas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4206 entries, 0 to 4205
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   TIPO_DE_SEGURO              4206 non-null   object        
 1   CERTIFICADO                 4206 non-null   object        
 2   NUMERO_INTERNO_DEL_CANAL    4206 non-null   object        
 3   TIPO_DE_REGISTRO            4206 non-null   object        
 4   MONEDA                      4206 non-null   object        
 5   TIPO_DE_MOVIMIENTO          4206 non-null   object        
 6   FECHA_DE_AFILIACION         0 non-null      datetime64[ns]
 7   FECHA_DE_INICIO_DEL_SEGURO  4206 non-null   datetime64[ns]
 8   FECHA_FIN_DEL_SEGURO        4206 non-null   datetime64[ns]
 9   PERIODO_DE_PAGO             4206 non-null   object        
 10  PRIMA                       4206 non-null   object        
 11  TRAMA_ORIGINAL              4206 non-null   object      